# Train YOLOv8s con kortxovision dataset

Ejecutar en Colab. Primero clonar el repo:
- `!git clone https://github.com/TKNIKA/kortxovision.git`
- `%cd kortxovision`

In [ ]:
# Instalar dependencias
!pip install -q ultralytics huggingface_hub

In [ ]:
# Descargar dataset desde HuggingFace (usa caché si ya existe)
from huggingface_hub import snapshot_download

dataset_path = snapshot_download(
    repo_id="mikeldiez/kortxovision_dataset",
    repo_type="dataset"
)

print(f"Dataset en: {dataset_path}")

In [ ]:
# Localizar data.yaml
from pathlib import Path

data_yaml = Path(dataset_path) / "data.yaml"
print(f"Usando data.yaml: {data_yaml}")

In [ ]:
# Entrenar YOLOv8s
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(data_yaml),
    epochs=10,
    imgsz=640,
    batch=16,
    name='kortxovision_yolov8s'
)

In [ ]:
# Evaluar y mostrar mAP50
metrics = model.val()
print(f"\nmAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

In [ ]:
# Probar inferencia con imágenes de test
from IPython.display import display
from PIL import Image as PILImage

test_images = ['test_imgs/test1.jpg', 'test_imgs/test2.jpg']

for img_path in test_images:
    print(f"\n{'='*50}")
    print(f"Inferencia en: {img_path}")
    print('='*50)
    
    # Realizar detección
    results = model(img_path)
    
    # Mostrar imagen con detecciones
    annotated = results[0].plot()
    annotated_img = PILImage.fromarray(annotated[..., ::-1])  # BGR to RGB
    display(annotated_img)
    
    # Mostrar detecciones
    if len(results[0].boxes) > 0:
        print(f"\nDetecciones: {len(results[0].boxes)}")
        for box in results[0].boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            print(f"  - {model.names[cls]}: {conf:.3f}")
    else:
        print("\nNo se detectaron objetos")

## Entrenamiento avanzado con más hiperparámetros

Si quieres ajustar más parámetros:

```python
model = YOLO('yolov8s.pt')

results = model.train(
    data=str(data_yaml),
    epochs=50,
    imgsz=640,
    batch=16,
    lr0=0.01,              # learning rate inicial
    lrf=0.01,              # learning rate final
    momentum=0.937,        # SGD momentum
    weight_decay=0.0005,   # optimizer weight decay
    warmup_epochs=3.0,     # warmup epochs
    warmup_momentum=0.8,   # warmup momentum
    box=7.5,               # box loss gain
    cls=0.5,               # cls loss gain
    dfl=1.5,               # dfl loss gain
    hsv_h=0.015,           # image HSV-Hue augmentation
    hsv_s=0.7,             # image HSV-Saturation augmentation
    hsv_v=0.4,             # image HSV-Value augmentation
    degrees=0.0,           # image rotation (+/- deg)
    translate=0.1,         # image translation (+/- fraction)
    scale=0.5,             # image scale (+/- gain)
    shear=0.0,             # image shear (+/- deg)
    perspective=0.0,       # image perspective (+/- fraction)
    flipud=0.0,            # image flip up-down (probability)
    fliplr=0.5,            # image flip left-right (probability)
    mosaic=1.0,            # image mosaic (probability)
    mixup=0.0,             # image mixup (probability)
    patience=50,           # epochs to wait for no improvement
    save_period=-1,        # save checkpoint every x epochs (-1 = disabled)
    project='runs/train',  # save results to project/name
    name='kortxovision_advanced',
    exist_ok=False,        # existing project/name ok, do not increment
    device=0,              # cuda device, i.e. 0 or 0,1,2,3 or cpu
    workers=8,             # number of worker threads for data loading
    optimizer='SGD',       # optimizer (SGD, Adam, AdamW, RMSProp)
    verbose=True,          # verbose output
    seed=0,                # random seed
    cos_lr=False,          # use cosine LR scheduler
    close_mosaic=10,       # disable mosaic augmentation for final epochs
    amp=True,              # Automatic Mixed Precision (AMP) training
)
```